# T1.1 — GEE Auth + Ortam Kurulumu (Colab)
Pomzadoya / Modul A v2 — P1 Veri Mühendisi

Bu notebook Google Earth Engine yetkilendirmesini ve Sentinel-2/S1/DEM çekimi için gerekli Python ortamını kurar.

**Süre:** ~10 dk (auth dahil)

**Çıktı:** Aktif `ee` oturumu, repo köküne göre `data/` klasörü hazır.


## 1. Bağımlılık kurulumu

In [ ]:
!pip install -q earthengine-api==0.1.384 rasterio==1.3.10 geopandas==0.14.4 shapely==2.0.4 numpy==1.26.4 scipy==1.13.1
!pip install -q gdal --no-deps || apt-get -qq install -y gdal-bin python3-gdal
import ee, geopandas as gpd, rasterio, numpy as np
print('ee:', ee.__version__, 'gpd:', gpd.__version__, 'rio:', rasterio.__version__)

## 2. Service Account JSON yükleme
GCP Console > IAM > Service Accounts > Earth Engine kullanıcısı için JSON key indir.
Aşağıdaki hücre çalıştığında dosya seçim diyaloğu açılır — `pomzadoya-sa.json` yükle.

In [ ]:
from google.colab import files
uploaded = files.upload()
import json, os
sa_path = list(uploaded.keys())[0]
with open(sa_path) as f:
    sa = json.load(f)
SERVICE_ACCOUNT = sa['client_email']
KEY_FILE = os.path.abspath(sa_path)
print('service account:', SERVICE_ACCOUNT)

## 3. GEE auth

In [ ]:
credentials = ee.ServiceAccountCredentials(SERVICE_ACCOUNT, KEY_FILE)
ee.Initialize(credentials, project=sa.get('project_id'))
print('GEE init OK — project:', sa.get('project_id'))
print('test query:', ee.Number(42).getInfo())

## 4. Repo bağlama (Drive mount + klasör)
Repo Drive'da: `/MyDrive/Pomzadoya/`. data/ alt klasörlerini oluştur.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
REPO = '/content/drive/MyDrive/Pomzadoya'
os.makedirs(f'{REPO}/data/aoi', exist_ok=True)
os.makedirs(f'{REPO}/data/s2_raw', exist_ok=True)
os.makedirs(f'{REPO}/data/ard', exist_ok=True)
os.makedirs(f'{REPO}/data/s1_stack', exist_ok=True)
os.makedirs(f'{REPO}/data/landsat', exist_ok=True)
os.makedirs(f'{REPO}/data/tiles', exist_ok=True)
print('repo:', REPO, '— alt klasörler hazır')

## 5. Sanity check
Avanos merkezinde küçük bir S2 görüntüsü çek, bant sayısı ve CRS bilgisini yazdır.

In [ ]:
pt = ee.Geometry.Point([34.85, 38.72])
img = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
       .filterBounds(pt)
       .filterDate('2024-07-01','2024-09-30')
       .sort('CLOUDY_PIXEL_PERCENTAGE')
       .first())
info = img.getInfo()
print('S2 sanity — id:', info['id'])
print('bands:', [b['id'] for b in info['bands']][:13])